In [1]:
import pandas as pd

bureau_balance = pd.read_csv("home-credit-default-risk/bureau_balance.csv")

In [2]:
missing_rate_percent = (bureau_balance.isnull().mean() * 100).sort_values(ascending=False)
print(missing_rate_percent)

SK_ID_BUREAU      0.0
MONTHS_BALANCE    0.0
STATUS            0.0
dtype: float64


In [3]:
status_counts = bureau_balance["STATUS"].value_counts()
print(status_counts)

STATUS
C    13646993
0     7499507
X     5810482
1      242347
5       62406
2       23419
3        8924
4        5847
Name: count, dtype: int64


In [4]:
# Build concise status indicators per customer

status_map = {
    "C": "Closed",
    "X": "Unknown",
    "0": "No_DPD",
    "1": "DPD_1_30",
    "2": "DPD_31_60",
    "3": "DPD_61_90",
    "4": "DPD_91_120",
    "5": "DPD_120_plus",
}

bureau_balance["STATUS_GROUP"] = bureau_balance["STATUS"].map(status_map)

status_summary = (
    bureau_balance.groupby(["SK_ID_BUREAU", "STATUS_GROUP"])["MONTHS_BALANCE"]
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

expected_groups = [
    "Closed",
    "Unknown",
    "No_DPD",
    "DPD_1_30",
    "DPD_31_60",
    "DPD_61_90",
    "DPD_91_120",
    "DPD_120_plus",
]
for col in expected_groups:
    if col not in status_summary.columns:
        status_summary[col] = 0

status_summary = status_summary[["SK_ID_BUREAU"] + expected_groups]

status_summary["bb_record_count"] = status_summary[expected_groups].sum(axis=1)

bb_months = (
    bureau_balance.groupby("SK_ID_BUREAU")["MONTHS_BALANCE"]
    .agg(["min", "max"])
    .reset_index()
    .rename(columns={"min": "bb_months_balance_min", "max": "bb_months_balance_max"})
)

bureau_balance_agg = status_summary.merge(bb_months, on="SK_ID_BUREAU", how="left")

bureau_ids = pd.read_csv(
    "home-credit-default-risk/bureau.csv", usecols=["SK_ID_BUREAU", "SK_ID_CURR"]
)

bureau_balance_agg = bureau_balance_agg.merge(bureau_ids, on="SK_ID_BUREAU", how="left")

agg_map = {col: "sum" for col in expected_groups + ["bb_record_count"]}
agg_map.update({"bb_months_balance_min": "min", "bb_months_balance_max": "max"})

bureau_balance_features = (
    bureau_balance_agg.groupby("SK_ID_CURR")
    .agg(agg_map)
    .reset_index()
)

rename_map = {
    "Closed": "bb_months_closed",
    "Unknown": "bb_months_unknown",
    "No_DPD": "bb_months_no_dpd",
    "DPD_1_30": "bb_months_dpd_1_30",
    "DPD_31_60": "bb_months_dpd_31_60",
    "DPD_61_90": "bb_months_dpd_61_90",
    "DPD_91_120": "bb_months_dpd_91_120",
    "DPD_120_plus": "bb_months_dpd_120_plus",
}

bureau_balance_features = bureau_balance_features.rename(columns=rename_map)

bureau_balance_features["bb_months_dpd_31_90"] = (
    bureau_balance_features["bb_months_dpd_31_60"]
    + bureau_balance_features["bb_months_dpd_61_90"]
)
bureau_balance_features["bb_months_dpd_90_plus"] = (
    bureau_balance_features["bb_months_dpd_91_120"]
    + bureau_balance_features["bb_months_dpd_120_plus"]
)

bureau_balance_features["bb_months_with_dpd"] = (
    bureau_balance_features["bb_months_dpd_1_30"]
    + bureau_balance_features["bb_months_dpd_31_90"]
    + bureau_balance_features["bb_months_dpd_90_plus"]
)

bureau_balance_features["bb_dpd_rate"] = (
    bureau_balance_features["bb_months_with_dpd"]
    / bureau_balance_features["bb_record_count"].replace(0, pd.NA)
).fillna(0)

bureau_balance_features = bureau_balance_features.drop(
    columns=[
        "bb_months_dpd_31_60",
        "bb_months_dpd_61_90",
        "bb_months_dpd_91_120",
        "bb_months_dpd_120_plus",
    ],
    errors="ignore",
)

bureau_balance_features.head()

,SK_ID_CURR,bb_months_closed,bb_months_unknown,bb_months_no_dpd,bb_months_dpd_1_30,bb_record_count,bb_months_balance_min,bb_months_balance_max,bb_months_dpd_31_90,bb_months_dpd_90_plus,bb_months_with_dpd,bb_dpd_rate
0,100001.0,110,30,31,1,172,-51,0,0,0,1,0.005814
1,100002.0,23,15,45,27,110,-47,0,0,0,27,0.245455
2,100005.0,5,2,14,0,21,-12,0,0,0,0,0.000000
3,100010.0,52,0,20,0,72,-90,-2,0,0,0,0.000000
4,100013.0,103,41,79,7,230,-68,0,0,0,7,0.030435


In [ ]:
pd.DataFrame.to_csv(bureau_balance_features,"transformed_data/_bureau_balance.csv") 